In [ ]:
import pandas as pd

def split_region_name(region):
    sido_list = [
        "서울", "부산", "대구", "인천", "광주", "대전", "울산",
        "세종", "경기도", "강원도", "충청북도", "충청남도",
        "경상북도", "경상남도", "전라북도", "전라남도",
        "제주특별자치도", "제주"
    ]

    for sido in sido_list:
        if region.startswith(sido):
            return sido + " " + region[len(sido):]
    return region

crime = pd.read_csv("crime_2023_before.csv", encoding="cp949")

region_cols = crime.columns[2:]

new_region_cols = [split_region_name(col) for col in region_cols]

rename_dict = dict(zip(region_cols, new_region_cols))

crime = crime.rename(columns=rename_dict)

crime.to_csv("crime_2023.csv", index=False, encoding='utf-8-sig')


In [3]:
import pandas as pd
import numpy as np

df = pd.read_excel("drinking_2023_before.xlsx")

df.columns = ["sido", "sigungu", "sub", "respond", "rate", "rate_se", "std_rate", "std_rate_se"]

df[["sido", "sigungu"]] = df[["sido", "sigungu"]].ffill()

sido_map = {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종",
    "경기도": "경기도", "강원특별자치도": "강원도",
    "충청북도": "충북", "충청남도": "충남",
    "전북특별자치도": "전북", "전라남도": "전남",
    "경상북도": "경북", "경상남도": "경남",
    "제주특별자치도": "제주"
}

df["sido_simple"] = df["sido"].map(sido_map)

df["region"] = df["sido_simple"] + " " + df["sigungu"]

def wmean(group, col):
    return np.average(group[col], weights=group["respond"])

result_rows = []

for region, g in df.groupby("region"):


    if (g["sub"].notna().any()) and ((g["sub"] != "소계").any()):

        sub_group = g[g["sub"] != "소계"]
        result_rows.append({
            "region": region,
            "respond": sub_group["respond"].sum(),
            "rate": wmean(sub_group, "rate"),
            "rate_se": wmean(sub_group, "rate_se"),
            "std_rate": wmean(sub_group, "std_rate"),
            "std_rate_se": wmean(sub_group, "std_rate_se")
        })
    else:
        for idx, row in g.iterrows():
            result_rows.append({
                "region": region,
                "respond": row["respond"],
                "rate": row["rate"],
                "rate_se": row["rate_se"],
                "std_rate": row["std_rate"],
                "std_rate_se": row["std_rate_se"]
            })

result = pd.DataFrame(result_rows)
result.to_excel("drinking_2023.xlsx", index=False)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("drinking_highrisk_2023_before.xlsx")

df.columns = ["sido", "sigungu", "sub", "respond", "rate", "rate_se", "std_rate", "std_rate_se"]

df[["sido", "sigungu"]] = df[["sido", "sigungu"]].ffill()

sido_map = {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종",
    "경기도": "경기도", "강원특별자치도": "강원도",
    "충청북도": "충북", "충청남도": "충남",
    "전북특별자치도": "전북", "전라남도": "전남",
    "경상북도": "경북", "경상남도": "경남",
    "제주특별자치도": "제주"
}

df["sido_simple"] = df["sido"].map(sido_map)

df["region"] = df["sido_simple"] + " " + df["sigungu"]

def wmean(group, col):
    return np.average(group[col], weights=group["respond"])

result_rows = []

for region, g in df.groupby("region"):


    if (g["sub"].notna().any()) and ((g["sub"] != "소계").any()):

        sub_group = g[g["sub"] != "소계"]
        result_rows.append({
            "region": region,
            "respond": sub_group["respond"].sum(),
            "rate": wmean(sub_group, "rate"),
            "rate_se": wmean(sub_group, "rate_se"),
            "std_rate": wmean(sub_group, "std_rate"),
            "std_rate_se": wmean(sub_group, "std_rate_se")
        })
    else:
        for idx, row in g.iterrows():
            result_rows.append({
                "region": region,
                "respond": row["respond"],
                "rate": row["rate"],
                "rate_se": row["rate_se"],
                "std_rate": row["std_rate"],
                "std_rate_se": row["std_rate_se"]
            })

result = pd.DataFrame(result_rows)
result.to_excel("drinking_highrisk_2023.xlsx", index=False)

In [ ]:
import pandas as pd
import re

sido_map = {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종",
    "경기도": "경기도", "강원도": "강원도", "충청북도": "충북",
    "충청남도": "충남", "전라북도": "전북", "전라남도": "전남",
    "경상북도": "경북", "경상남도": "경남",
    "제주특별자치도": "제주"
}

sido_pattern = re.compile(r"(특별시|광역시|특별자치시|특별자치도)")

df = pd.read_excel("population_before.xlsx")
df.columns = ["region", "total", "male", "female"]

output_rows = []
current_sido = None

for idx, row in df.iterrows():
    name = str(row["region"]).strip()

    if name == "nan" or name == "":
        continue

    if sido_pattern.search(name):
        if name in sido_map:
            current_sido = sido_map[name]
        else:
            current_sido = name
        continue

    if name == "전국":
        continue

    full_name = f"{current_sido} {name}"

    output_rows.append({
        "행정구역명": full_name,
        "총인구수": row["total"],
        "남": row["male"],
        "여": row["female"]
    })

result = pd.DataFrame(output_rows)
result.to_excel("population.xlsx", index=False)
